In [ ]:
!pip -q install jupysql duckdb-engine

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/ecommerce-growth-marketing-analytics'
CLEAN_PATH = f'{PROJECT_PATH}/data/cleaned'

print(PROJECT_PATH)

In [ ]:
%load_ext sql

In [ ]:
%sql duckdb:///:memory:

In [ ]:
%%sql

SELECT 1 AS test_value;

In [ ]:
%%sql

CREATE TABLE orders AS
SELECT *
FROM read_csv_auto(
    '/content/drive/MyDrive/ecommerce-growth-marketing-analytics/data/cleaned/olist_orders_cleaned.csv',
    header = true
);

In [ ]:
%%sql

SELECT COUNT(*) AS total_rows
FROM orders;

In [ ]:
%config SqlMagic.displaylimit = 40

In [ ]:
%%sql

SELECT
    column_name,
    data_type
FROM information_schema.columns
WHERE table_name = 'orders'
ORDER BY ordinal_position;

In [ ]:
# customer column loading

In [ ]:
%%sql

CREATE TABLE customers AS
SELECT *
FROM read_csv_auto(
    '/content/drive/MyDrive/ecommerce-growth-marketing-analytics/data/cleaned/olist_customers_cleaned.csv',
    header = true
);

In [ ]:
%%sql

SELECT COUNT(*) AS total_rows
FROM customers;

In [ ]:
%%sql

CREATE TABLE order_items AS
SELECT *
FROM read_csv_auto(
    '/content/drive/MyDrive/ecommerce-growth-marketing-analytics/data/cleaned/olist_order_items_cleaned.csv',
    header = true
);

In [ ]:
%%sql

SELECT COUNT(*) AS total_rows
FROM order_items;

In [ ]:
%%sql

CREATE TABLE payments AS
SELECT *
FROM read_csv_auto(
    '/content/drive/MyDrive/ecommerce-growth-marketing-analytics/data/cleaned/olist_order_payments_cleaned.csv',
    header = true
);

In [ ]:
%%sql

SELECT COUNT(*) AS total_rows
FROM payments;

In [ ]:
%%sql

CREATE TABLE reviews AS
SELECT *
FROM read_csv_auto(
    '/content/drive/MyDrive/ecommerce-growth-marketing-analytics/data/cleaned/olist_order_reviews_cleaned.csv',
    header = true
);

In [ ]:
%%sql

SELECT COUNT(*) AS total_rows
FROM reviews;

In [ ]:
%%sql

CREATE TABLE products AS
SELECT *
FROM read_csv_auto(
    '/content/drive/MyDrive/ecommerce-growth-marketing-analytics/data/cleaned/olist_products_cleaned.csv',
    header = true
);

In [ ]:
%%sql

SELECT COUNT(*) AS total_rows
FROM products;

In [ ]:
%%sql

CREATE TABLE sellers AS
SELECT *
FROM read_csv_auto(
    '/content/drive/MyDrive/ecommerce-growth-marketing-analytics/data/cleaned/olist_sellers_cleaned.csv',
    header = true
);

In [ ]:
%%sql

SELECT COUNT(*) AS total_rows
FROM sellers;

In [ ]:
%%sql

CREATE TABLE category_translation AS
SELECT *
FROM read_csv_auto(
    '/content/drive/MyDrive/ecommerce-growth-marketing-analytics/data/cleaned/product_category_name_translation_cleaned.csv',
    header = true
);

In [ ]:
%%sql

SELECT COUNT(*) AS total_rows
FROM category_translation;

In [ ]:
%%sql

CREATE TABLE geolocation AS
SELECT *
FROM read_csv_auto(
    '/content/drive/MyDrive/ecommerce-growth-marketing-analytics/data/cleaned/olist_geolocation_cleaned.csv',
    header = true,
    quote = '"'
);

In [ ]:
%%sql

SELECT COUNT(*) AS total_rows
FROM geolocation;

In [ ]:
%%sql

SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'main'
ORDER BY table_name;

In [ ]:
%%sql

SELECT
    ROUND(SUM(price), 2) AS total_product_sales
FROM order_items;

In [ ]:
%%sql

SELECT
    o.purchase_year_month,
    ROUND(SUM(oi.price), 2) AS monthly_product_sales
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY o.purchase_year_month
ORDER BY o.purchase_year_month;

In [ ]:
%%sql

SELECT
    purchase_year_month,
    COUNT(*) AS total_orders
FROM orders
GROUP BY purchase_year_month
ORDER BY purchase_year_month;

In [ ]:
%%sql

SELECT
    o.purchase_year_month,
    ROUND(SUM(oi.price) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY o.purchase_year_month
ORDER BY o.purchase_year_month;

In [ ]:
%%sql

SELECT
    ROUND(
        SUM(oi.price) / COUNT(DISTINCT oi.order_id),
        2
    ) AS overall_avg_order_value
FROM order_items oi;

In [ ]:
%%sql

SELECT
    o.order_status,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price), 2) AS product_sales
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY o.order_status
ORDER BY product_sales DESC;

In [ ]:
%%sql

SELECT
    ROUND(
        100.0 * SUM(CASE
            WHEN o.order_status = 'delivered' THEN oi.price
            ELSE 0
        END) / SUM(oi.price),
        2
    ) AS delivered_sales_share_pct
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id;

In [ ]:
%%sql

SELECT
    ROUND(
        SUM(oi.price) / COUNT(DISTINCT o.order_id),
        2
    ) AS delivered_avg_order_value
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered';

In [ ]:
## Business Question 2 — Product Category Performance

In [ ]:
%%sql

SELECT
    ct.product_category_name_english AS category,
    ROUND(SUM(oi.price), 2) AS product_sales
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
LEFT JOIN category_translation ct
    ON p.product_category_name = ct.product_category_name
GROUP BY ct.product_category_name_english
ORDER BY product_sales DESC;

In [ ]:
%%sql

SELECT
    COALESCE(
        ct.product_category_name_english,
        'Unknown / Untranslated'
    ) AS category,

    ROUND(SUM(oi.price), 2) AS delivered_product_sales

FROM order_items oi

JOIN orders o
    ON oi.order_id = o.order_id

JOIN products p
    ON oi.product_id = p.product_id

LEFT JOIN category_translation ct
    ON p.product_category_name = ct.product_category_name

WHERE o.order_status = 'delivered'

GROUP BY
    COALESCE(
        ct.product_category_name_english,
        'Unknown / Untranslated'
    )

ORDER BY delivered_product_sales DESC;

In [ ]:
%%sql

SELECT
    COALESCE(
        ct.product_category_name_english,
        'Unknown / Untranslated'
    ) AS category,

    COUNT(DISTINCT oi.order_id) AS delivered_orders

FROM order_items oi

JOIN orders o
    ON oi.order_id = o.order_id

JOIN products p
    ON oi.product_id = p.product_id

LEFT JOIN category_translation ct
    ON p.product_category_name = ct.product_category_name

WHERE o.order_status = 'delivered'

GROUP BY
    COALESCE(
        ct.product_category_name_english,
        'Unknown / Untranslated'
    )

ORDER BY delivered_orders DESC;

In [ ]:
%%sql

SELECT
    COALESCE(
        ct.product_category_name_english,
        'Unknown / Untranslated'
    ) AS category,

    ROUND(
        SUM(oi.price) / COUNT(DISTINCT oi.order_id),
        2
    ) AS avg_category_order_value

FROM order_items oi

JOIN orders o
    ON oi.order_id = o.order_id

JOIN products p
    ON oi.product_id = p.product_id

LEFT JOIN category_translation ct
    ON p.product_category_name = ct.product_category_name

WHERE o.order_status = 'delivered'

GROUP BY
    COALESCE(
        ct.product_category_name_english,
        'Unknown / Untranslated'
    )

ORDER BY avg_category_order_value DESC;

In [ ]:
%%sql

SELECT
    COALESCE(
        ct.product_category_name_english,
        'Unknown / Untranslated'
    ) AS category,

    ROUND(SUM(oi.price), 2) AS delivered_product_sales,

    ROUND(
        100.0 * SUM(oi.price)
        / SUM(SUM(oi.price)) OVER (),
        2
    ) AS sales_share_pct

FROM order_items oi

JOIN orders o
    ON oi.order_id = o.order_id

JOIN products p
    ON oi.product_id = p.product_id

LEFT JOIN category_translation ct
    ON p.product_category_name = ct.product_category_name

WHERE o.order_status = 'delivered'

GROUP BY
    COALESCE(
        ct.product_category_name_english,
        'Unknown / Untranslated'
    )

ORDER BY delivered_product_sales DESC

LIMIT 10;

In [ ]:
%%sql

SELECT
    COALESCE(
        ct.product_category_name_english,
        'Unknown / Untranslated'
    ) AS category,

    ROUND(SUM(oi.price), 2) AS delivered_product_sales,

    COUNT(DISTINCT oi.order_id) AS delivered_orders

FROM order_items oi

JOIN orders o
    ON oi.order_id = o.order_id

JOIN products p
    ON oi.product_id = p.product_id

LEFT JOIN category_translation ct
    ON p.product_category_name = ct.product_category_name

WHERE o.order_status = 'delivered'

GROUP BY
    COALESCE(
        ct.product_category_name_english,
        'Unknown / Untranslated'
    )

ORDER BY delivered_product_sales ASC

LIMIT 10;

In [ ]:
%%sql

SELECT
    ROUND(SUM(oi.price), 2) AS total_delivered_product_sales

FROM order_items oi

JOIN orders o
    ON oi.order_id = o.order_id

JOIN products p
    ON oi.product_id = p.product_id

LEFT JOIN category_translation ct
    ON p.product_category_name = ct.product_category_name

WHERE o.order_status = 'delivered';

In [ ]:
## Business Question 2 — What do customers' purchasing patterns look like?

In [ ]:
%%sql

SELECT
    COUNT(DISTINCT customer_unique_id) AS unique_customers
FROM customers;

In [ ]:
%%sql

SELECT
    customer_unique_id,
    COUNT(*) AS total_orders
FROM customers
GROUP BY customer_unique_id
HAVING COUNT(*) > 1
ORDER BY total_orders DESC
LIMIT 10;

In [ ]:
%%sql

WITH customer_orders AS (
    SELECT
        customer_unique_id,
        COUNT(*) AS total_orders
    FROM customers
    GROUP BY customer_unique_id
)

SELECT
    COUNT(*) AS total_customers,

    COUNT(
        CASE
            WHEN total_orders > 1 THEN 1
        END
    ) AS repeat_customers,

    ROUND(
        100.0 * COUNT(
            CASE
                WHEN total_orders > 1 THEN 1
            END
        ) / COUNT(*),
        2
    ) AS repeat_customer_rate_pct

FROM customer_orders;

In [ ]:
%%sql

WITH customer_orders AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS total_delivered_orders

    FROM customers c

    JOIN orders o
        ON c.customer_id = o.customer_id

    WHERE o.order_status = 'delivered'

    GROUP BY c.customer_unique_id
)

SELECT
    COUNT(*) AS total_delivered_customers,

    COUNT(
        CASE
            WHEN total_delivered_orders > 1 THEN 1
        END
    ) AS repeat_delivered_customers,

    ROUND(
        100.0 * COUNT(
            CASE
                WHEN total_delivered_orders > 1 THEN 1
            END
        ) / COUNT(*),
        2
    ) AS repeat_delivered_customer_rate_pct

FROM customer_orders;

In [ ]:
%%sql

WITH customer_orders AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS total_delivered_orders

    FROM customers c

    JOIN orders o
        ON c.customer_id = o.customer_id

    WHERE o.order_status = 'delivered'

    GROUP BY c.customer_unique_id
)

SELECT
    total_delivered_orders,
    COUNT(*) AS number_of_customers

FROM customer_orders

GROUP BY total_delivered_orders

ORDER BY total_delivered_orders;

In [ ]:
%%sql

WITH customer_orders AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS total_delivered_orders

    FROM customers c

    JOIN orders o
        ON c.customer_id = o.customer_id

    WHERE o.order_status = 'delivered'

    GROUP BY c.customer_unique_id
)

SELECT
    ROUND(
        AVG(total_delivered_orders),
        2
    ) AS avg_delivered_orders_per_customer

FROM customer_orders;

In [ ]:
%%sql

WITH customer_spending AS (
    SELECT
        c.customer_unique_id,
        SUM(oi.price) AS total_product_spending

    FROM customers c

    JOIN orders o
        ON c.customer_id = o.customer_id

    JOIN order_items oi
        ON o.order_id = oi.order_id

    WHERE o.order_status = 'delivered'

    GROUP BY c.customer_unique_id
)

SELECT
    ROUND(AVG(total_product_spending), 2)
        AS avg_product_spending_per_customer

FROM customer_spending;

In [ ]:
%%sql

WITH customer_summary AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS total_delivered_orders,
        SUM(oi.price) AS total_product_spending

    FROM customers c

    JOIN orders o
        ON c.customer_id = o.customer_id

    JOIN order_items oi
        ON o.order_id = oi.order_id

    WHERE o.order_status = 'delivered'

    GROUP BY c.customer_unique_id
)

SELECT
    CASE
        WHEN total_delivered_orders = 1 THEN 'One-time customers'
        ELSE 'Repeat customers'
    END AS customer_type,

    COUNT(*) AS number_of_customers,

    ROUND(SUM(total_product_spending), 2)
        AS total_product_spending,

    ROUND(AVG(total_product_spending), 2)
        AS avg_spending_per_customer

FROM customer_summary

GROUP BY customer_type

ORDER BY avg_spending_per_customer DESC;

In [ ]:
%%sql

WITH customer_summary AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS total_delivered_orders,
        SUM(oi.price) AS total_product_spending

    FROM customers c

    JOIN orders o
        ON c.customer_id = o.customer_id

    JOIN order_items oi
        ON o.order_id = oi.order_id

    WHERE o.order_status = 'delivered'

    GROUP BY c.customer_unique_id
)

SELECT
    ROUND(
        100.0 * SUM(
            CASE
                WHEN total_delivered_orders > 1
                THEN total_product_spending
                ELSE 0
            END
        ) / SUM(total_product_spending),
        2
    ) AS repeat_customer_sales_share_pct

FROM customer_summary;

In [ ]:
%%sql

SELECT
    ROUND(SUM(oi.price), 2) AS total_delivered_product_sales

FROM customers c

JOIN orders o
    ON c.customer_id = o.customer_id

JOIN order_items oi
    ON o.order_id = oi.order_id

WHERE o.order_status = 'delivered';

In [ ]:
## Business Question 4 — Geographic Sales Performance

In [ ]:
%%sql

SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS delivered_orders,
    ROUND(SUM(oi.price), 2) AS delivered_product_sales

FROM customers c

JOIN orders o
    ON c.customer_id = o.customer_id

JOIN order_items oi
    ON o.order_id = oi.order_id

WHERE o.order_status = 'delivered'

GROUP BY c.customer_state

ORDER BY delivered_product_sales DESC;

In [ ]:
%%sql

SELECT
    c.customer_state,

    COUNT(DISTINCT o.order_id) AS delivered_orders,

    ROUND(
        SUM(oi.price) / COUNT(DISTINCT o.order_id),
        2
    ) AS avg_order_value

FROM customers c

JOIN orders o
    ON c.customer_id = o.customer_id

JOIN order_items oi
    ON o.order_id = oi.order_id

WHERE o.order_status = 'delivered'

GROUP BY c.customer_state

ORDER BY avg_order_value DESC;

In [ ]:
%%sql

SELECT
    c.customer_state,

    ROUND(SUM(oi.price), 2) AS delivered_product_sales,

    ROUND(
        100.0 * SUM(oi.price)
        / SUM(SUM(oi.price)) OVER (),
        2
    ) AS sales_share_pct

FROM customers c

JOIN orders o
    ON c.customer_id = o.customer_id

JOIN order_items oi
    ON o.order_id = oi.order_id

WHERE o.order_status = 'delivered'

GROUP BY c.customer_state

ORDER BY delivered_product_sales DESC;

In [ ]:
%%sql

SELECT
    c.customer_state,
    c.customer_city,

    COUNT(DISTINCT o.order_id) AS delivered_orders,

    ROUND(SUM(oi.price), 2) AS delivered_product_sales

FROM customers c

JOIN orders o
    ON c.customer_id = o.customer_id

JOIN order_items oi
    ON o.order_id = oi.order_id

WHERE o.order_status = 'delivered'

GROUP BY
    c.customer_state,
    c.customer_city

ORDER BY delivered_product_sales DESC

LIMIT 10;

In [ ]:
%%sql

SELECT
    ROUND(SUM(oi.price), 2) AS total_geographic_sales

FROM customers c

JOIN orders o
    ON c.customer_id = o.customer_id

JOIN order_items oi
    ON o.order_id = oi.order_id

WHERE o.order_status = 'delivered';

In [ ]:
## Business Question 5 — Delivery Performance

In [ ]:
%%sql

SELECT
    order_status,
    COUNT(*) AS total_orders
FROM orders
GROUP BY order_status
ORDER BY total_orders DESC;

In [ ]:
%%sql

SELECT
    COUNT(*) AS delivered_orders_with_date,

    SUM(
        CASE
            WHEN is_late = TRUE THEN 1
            ELSE 0
        END
    ) AS late_orders,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN is_late = TRUE THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS late_delivery_rate_pct

FROM orders

WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL;

In [ ]:
%%sql

SELECT
    ROUND(AVG(delivery_time_days), 2) AS avg_delivery_time_days,
    MEDIAN(delivery_time_days) AS median_delivery_time_days

FROM orders

WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL;

In [ ]:
%%sql

SELECT
    CASE
        WHEN delivery_time_days <= 7 THEN '0–7 days'
        WHEN delivery_time_days <= 14 THEN '8–14 days'
        WHEN delivery_time_days <= 30 THEN '15–30 days'
        ELSE 'Over 30 days'
    END AS delivery_time_group,

    COUNT(*) AS total_orders

FROM orders

WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL
  AND delivery_time_days >= 0

GROUP BY delivery_time_group

ORDER BY
    CASE delivery_time_group
        WHEN '0–7 days' THEN 1
        WHEN '8–14 days' THEN 2
        WHEN '15–30 days' THEN 3
        ELSE 4
    END;

In [ ]:
%%sql

SELECT
    COUNT(*) AS late_orders,

    ROUND(AVG(delivery_delay_days), 2) AS avg_days_late,

    MEDIAN(delivery_delay_days) AS median_days_late

FROM orders

WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL
  AND is_late = TRUE;

In [ ]:
%%sql

SELECT
    c.customer_state,

    COUNT(*) AS delivered_orders_with_date,

    SUM(
        CASE
            WHEN o.is_late = TRUE THEN 1
            ELSE 0
        END
    ) AS late_orders,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.is_late = TRUE THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS late_delivery_rate_pct

FROM orders o

JOIN customers c
    ON o.customer_id = c.customer_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL

GROUP BY c.customer_state

ORDER BY late_delivery_rate_pct DESC;

In [ ]:
%%sql

SELECT
    c.customer_state,

    SUM(
        CASE
            WHEN o.is_late = TRUE THEN 1
            ELSE 0
        END
    ) AS late_orders,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.is_late = TRUE THEN 1
                ELSE 0
            END
        )
        / SUM(
            SUM(
                CASE
                    WHEN o.is_late = TRUE THEN 1
                    ELSE 0
                END
            )
        ) OVER (),
        2
    ) AS share_of_all_late_orders_pct

FROM orders o

JOIN customers c
    ON o.customer_id = c.customer_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL

GROUP BY c.customer_state

ORDER BY late_orders DESC;

In [ ]:
%%sql

SELECT
    COUNT(*) AS delivered_orders,

    SUM(
        CASE
            WHEN order_delivered_customer_date IS NULL THEN 1
            ELSE 0
        END
    ) AS missing_delivery_dates

FROM orders

WHERE order_status = 'delivered';

In [ ]:
## Business Question 6 — Customer Review Analysis

In [71]:
%%sql

SELECT
    review_score,
    COUNT(*) AS review_records

FROM reviews

GROUP BY review_score

ORDER BY review_score;

Running query in 'duckdb:///:memory:'

review_score,review_records
1,11424
2,3151
3,8179
4,19142
5,57328


In [72]:
%%sql

SELECT
    COUNT(*) AS total_review_records,

    ROUND(AVG(review_score), 2) AS avg_review_score

FROM reviews;

Running query in 'duckdb:///:memory:'

total_review_records,avg_review_score
99224,4.09


In [73]:
%%sql

SELECT
    COUNT(*) AS orders_with_reviews,

    SUM(
        CASE
            WHEN review_count > 1 THEN 1
            ELSE 0
        END
    ) AS orders_with_multiple_reviews

FROM (
    SELECT
        order_id,
        COUNT(*) AS review_count

    FROM reviews

    GROUP BY order_id
) AS review_counts;

Running query in 'duckdb:///:memory:'

orders_with_reviews,orders_with_multiple_reviews
98673,547


In [74]:
%%sql

SELECT
    order_id,
    COUNT(*) AS review_count,
    ROUND(AVG(review_score), 2) AS order_avg_review_score

FROM reviews

GROUP BY order_id

ORDER BY review_count DESC

LIMIT 10;

Running query in 'duckdb:///:memory:'

order_id,review_count,order_avg_review_score
03c939fd7fd3b38f8485a0f95798f1f6,3,3.33
c88b1d1b157a9999ce368f218a407141,3,4.33
df56136b8031ecd28e200bb18e6ddb2e,3,5.0
8e17072ec97ce29f0e1f111e598b0c85,3,1.0
fd61441ba2a7b57e6342862e779b10b0,2,4.5
2daee070f2042c8b7a8e9fdde778a31a,2,5.0
4703440eb9289d3769819920e98ec061,2,5.0
076d991923ee938761994a4255f5c5da,2,5.0
80446aee36e09ebf4cb79585f6c9ce61,2,5.0
45390ff93f092f0de1524d486a172c33,2,1.0


In [75]:
%%sql

WITH order_reviews AS (
    SELECT
        order_id,
        AVG(review_score) AS order_avg_review_score
    FROM reviews
    GROUP BY order_id
)

SELECT
    CASE
        WHEN o.is_late = TRUE THEN 'Late'
        ELSE 'Not late'
    END AS delivery_group,

    COUNT(*) AS reviewed_orders,

    ROUND(AVG(r.order_avg_review_score), 2) AS avg_review_score

FROM orders o

JOIN order_reviews r
    ON o.order_id = r.order_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL

GROUP BY delivery_group

ORDER BY delivery_group;

Running query in 'duckdb:///:memory:'

delivery_group,reviewed_orders,avg_review_score
Late,6381,2.27
Not late,89443,4.29


In [76]:
%%sql

WITH order_reviews AS (
    SELECT
        order_id,
        AVG(review_score) AS order_avg_review_score
    FROM reviews
    GROUP BY order_id
)

SELECT
    CASE
        WHEN o.is_late = TRUE THEN 'Late'
        ELSE 'Not late'
    END AS delivery_group,

    COUNT(*) AS reviewed_orders,

    SUM(
        CASE
            WHEN r.order_avg_review_score <= 2 THEN 1
            ELSE 0
        END
    ) AS low_rated_orders,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN r.order_avg_review_score <= 2 THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS low_rating_rate_pct

FROM orders o

JOIN order_reviews r
    ON o.order_id = r.order_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL

GROUP BY delivery_group

ORDER BY delivery_group;

Running query in 'duckdb:///:memory:'

delivery_group,reviewed_orders,low_rated_orders,low_rating_rate_pct
Late,6381,3979,62.36
Not late,89443,8257,9.23


In [77]:
%%sql

WITH reviewed_orders AS (
    SELECT DISTINCT
        order_id
    FROM reviews
)

SELECT
    COUNT(*) AS delivered_orders_with_date,

    COUNT(r.order_id) AS orders_with_reviews,

    COUNT(*) - COUNT(r.order_id) AS orders_without_reviews

FROM orders o

LEFT JOIN reviewed_orders r
    ON o.order_id = r.order_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL;

Running query in 'duckdb:///:memory:'

delivered_orders_with_date,orders_with_reviews,orders_without_reviews
96470,95824,646


In [78]:
%%sql

WITH order_reviews AS (
    SELECT
        order_id,
        AVG(review_score) AS order_avg_review_score
    FROM reviews
    GROUP BY order_id
)

SELECT
    CASE
        WHEN o.delivery_time_days <= 7 THEN '0–7 days'
        WHEN o.delivery_time_days <= 14 THEN '8–14 days'
        WHEN o.delivery_time_days <= 30 THEN '15–30 days'
        ELSE 'Over 30 days'
    END AS delivery_time_group,

    COUNT(*) AS reviewed_orders,

    ROUND(
        AVG(r.order_avg_review_score),
        2
    ) AS avg_review_score

FROM orders o

JOIN order_reviews r
    ON o.order_id = r.order_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.delivery_time_days >= 0

GROUP BY delivery_time_group

ORDER BY
    CASE delivery_time_group
        WHEN '0–7 days' THEN 1
        WHEN '8–14 days' THEN 2
        WHEN '15–30 days' THEN 3
        ELSE 4
    END;

Running query in 'duckdb:///:memory:'

delivery_time_group,reviewed_orders,avg_review_score
0–7 days,33539,4.41
8–14 days,36190,4.29
15–30 days,22088,3.91
Over 30 days,4007,2.18


In [84]:
%%sql

WITH order_reviews AS (
    SELECT
        order_id,
        AVG(review_score) AS order_avg_review_score
    FROM reviews
    GROUP BY order_id
),

order_categories AS (
    SELECT DISTINCT
        oi.order_id,
        COALESCE(
            ct.product_category_name_english,
            'Unknown / Untranslated'
        ) AS product_category
    FROM order_items oi

    JOIN products p
        ON oi.product_id = p.product_id

    LEFT JOIN category_translation ct
        ON p.product_category_name = ct.product_category_name
)

SELECT
    oc.product_category,
    COUNT(*) AS reviewed_orders,
    ROUND(AVG(r.order_avg_review_score), 2) AS avg_review_score

FROM order_categories oc

JOIN order_reviews r
    ON oc.order_id = r.order_id

JOIN orders o
    ON oc.order_id = o.order_id

WHERE o.order_status = 'delivered'

GROUP BY oc.product_category

HAVING COUNT(*) >= 100

ORDER BY avg_review_score ASC, reviewed_orders DESC;

Running query in 'duckdb:///:memory:'

product_category,reviewed_orders,avg_review_score
office_furniture,1244,3.64
fashion_male_clothing,105,3.82
audio,345,3.84
home_confort,390,3.89
fixed_telephony,209,3.97
construction_tools_safety,158,3.97
home_construction,481,3.99
bed_bath_table,9177,4.0
Unknown / Untranslated,1402,4.0
fashion_underwear_beach,116,4.01


In [85]:
%%sql

WITH office_furniture_orders AS (
    SELECT DISTINCT
        oi.order_id
    FROM order_items oi

    JOIN products p
        ON oi.product_id = p.product_id

    WHERE p.product_category_name = 'moveis_escritorio'
)

SELECT
    COUNT(*) AS delivered_orders_with_date,

    SUM(
        CASE
            WHEN o.is_late = TRUE THEN 1
            ELSE 0
        END
    ) AS late_orders,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.is_late = TRUE THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS late_delivery_rate_pct

FROM office_furniture_orders f

JOIN orders o
    ON f.order_id = o.order_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL;

Running query in 'duckdb:///:memory:'

delivered_orders_with_date,late_orders,late_delivery_rate_pct
1254,101,8.05


In [86]:
%%sql

WITH office_furniture_orders AS (
    SELECT DISTINCT
        oi.order_id
    FROM order_items oi

    JOIN products p
        ON oi.product_id = p.product_id

    WHERE p.product_category_name = 'moveis_escritorio'
),

order_reviews AS (
    SELECT
        order_id,
        AVG(review_score) AS order_avg_review_score
    FROM reviews
    GROUP BY order_id
)

SELECT
    CASE
        WHEN o.is_late = TRUE THEN 'Late'
        ELSE 'Not late'
    END AS delivery_group,

    COUNT(*) AS reviewed_orders,

    ROUND(
        AVG(r.order_avg_review_score),
        2
    ) AS avg_review_score

FROM office_furniture_orders f

JOIN orders o
    ON f.order_id = o.order_id

JOIN order_reviews r
    ON f.order_id = r.order_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL

GROUP BY delivery_group

ORDER BY delivery_group;

Running query in 'duckdb:///:memory:'

delivery_group,reviewed_orders,avg_review_score
Late,99,2.35
Not late,1145,3.75


In [87]:
%%sql

WITH office_furniture_orders AS (
    SELECT DISTINCT
        oi.order_id
    FROM order_items oi

    JOIN products p
        ON oi.product_id = p.product_id

    WHERE p.product_category_name = 'moveis_escritorio'
),

order_reviews AS (
    SELECT
        order_id,
        AVG(review_score) AS order_avg_review_score
    FROM reviews
    GROUP BY order_id
)

SELECT
    CASE
        WHEN o.is_late = TRUE THEN 'Late'
        ELSE 'Not late'
    END AS delivery_group,

    COUNT(*) AS reviewed_orders,

    ROUND(
        AVG(r.order_avg_review_score),
        2
    ) AS avg_review_score

FROM office_furniture_orders f

JOIN orders o
    ON f.order_id = o.order_id

JOIN order_reviews r
    ON f.order_id = r.order_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL

GROUP BY delivery_group

ORDER BY delivery_group;

Running query in 'duckdb:///:memory:'

delivery_group,reviewed_orders,avg_review_score
Late,99,2.35
Not late,1145,3.75


## Business Question 7 — Evidence-Based Business Recommendations

| Business area | Validated finding | Potential area to investigate |
|---|---|---|
| Customer purchasing | 3.00% of customers with delivered orders made repeat purchases during the observation period. | Understand barriers to repeat purchasing. |
| Delivery experience | Late orders averaged 2.27/5 in reviews, compared with 4.29/5 for not-late orders. | Investigate causes of late delivery and opportunities to improve punctuality. |
| Geographic delivery | Rio de Janeiro had a 12.11% late-delivery rate across 12,350 delivered orders with dates. | Investigate delivery processes affecting customers in Rio de Janeiro. |
| Product experience | Office furniture averaged 3.64/5 in reviews; not-late office-furniture orders averaged 3.75/5. | Investigate product, packaging and service feedback alongside delivery performance. |::\

## Final Business Insights

### Insight 1 — Sales are concentrated in a few locations

São Paulo state contributed 38.33% of delivered product sales,
followed by Rio de Janeiro (13.31%) and Minas Gerais (11.74%).

Together, these three states accounted for 63.38% of delivered
product sales in the historical Olist dataset.

São Paulo's high sales value was associated with its large order
volume: 40,501 delivered orders, with an average product sales
value of BRL 125.12 per order.

**Business implication:** Geographic sales concentration is
important when reviewing market performance. However, these
results alone do not establish which states offer the greatest
future growth potential.

### Insight 2 — Repeat purchasing is limited within the observation period

Of the 93,358 customers who placed at least one delivered order,
2,801 customers (3.00%) placed multiple delivered orders during
the dataset's observation period.

Repeat customers contributed 5.51% of delivered product sales.

Average total product spending per repeat customer was BRL 260.05,
compared with BRL 137.96 for one-time customers.

**Business implication:** Understanding repeat purchasing could
help the marketplace identify opportunities to improve customer
retention. However, the historical dataset does not capture
purchases outside its observation period, and these results alone
do not establish why customers did or did not purchase again.

### Insight 3 — Late deliveries are associated with lower customer ratings

Among 95,824 reviewed, delivered orders with recorded delivery
dates, late orders received an average review score of 2.27/5,
compared with 4.29/5 for orders classified as not late.

The percentage of orders with an average review score of 2 or
lower was 62.36% for late orders, compared with 9.23% for
not-late orders.

Review scores also differed by delivery duration. Orders delivered
within 7 days averaged 4.41/5, while orders taking over 30 days
averaged 2.18/5.

**Business implication:** Delivery performance is an important
area for further investigation when reviewing customer experience.
The observed relationships do not prove that delivery delays
caused lower review scores.

In [ ]:
### Insight 4 — Product sales are concentrated in a few categories

Health & Beauty generated the highest delivered product sales
(BRL 1,233,131.72), followed by Watches & Gifts
(BRL 1,166,176.98) and Bed, Bath & Table
(BRL 1,023,434.76).

The five highest-selling product categories together contributed
approximately 39.83% of total delivered product sales.

However, high product sales do not necessarily indicate high
profitability because product costs and operating expenses
are not available in the dataset.

**Business implication:** Category-level performance can help
the marketplace identify where sales are concentrated and
which categories warrant further analysis. Profitability and
future growth potential require additional data.

### Insight 5 — Office furniture has comparatively lower customer ratings

Office furniture received an average review score of 3.64/5
across 1,244 reviewed, delivered orders.

Its late-delivery rate was 8.05%, compared with the overall
marketplace rate of 6.77%.

Late office-furniture orders averaged 2.35/5 in reviews, while
office-furniture orders classified as not late averaged 3.75/5.

**Business implication:** Delivery punctuality may be one factor
associated with lower office-furniture ratings, but it does not
fully explain the difference. Customer feedback should be
examined for recurring complaints about product condition,
packaging, assembly or seller service before recommending
specific operational changes.

### Recommendation 1 — Investigate and reduce late deliveries

**Evidence:** Late orders received an average review score of 2.27/5, compared with 4.29/5 for orders classified as not late. Rio de Janeiro had a 12.11% late-delivery rate across 12,350 delivered orders with recorded delivery dates.

**Proposed action:** Review the delivery process for late orders, starting with Rio de Janeiro. Examine seller dispatch times, carrier performance and delivery routes to identify where delays occur.

**How to measure results:** Track the late-delivery rate, average days late and average review score before and after any operational changes.

**Limitation:** The data shows an association between late delivery and lower ratings. It does not prove that reducing late deliveries will increase review scores.

### Recommendation 2 — Investigate repeat purchasing

**Evidence:** Of the 93,358 customers with at least one delivered order, 2,801 (3.00%) made multiple delivered purchases during the dataset's observation period. Repeat customers contributed 5.51% of delivered product sales.

**Proposed action:** Survey a sample of customers after their first purchase to understand their experience and whether they intend to buy again. Examine repeat purchasing by product category to distinguish products bought frequently from products typically bought occasionally.

**How to measure results:** Track the percentage of first-time customers who place a second delivered order within a defined period, such as 90 days. Compare customers who have had a full 90-day observation window.

**Limitation:** The historical dataset does not capture purchases outside its observation period. A low observed repeat-purchase rate does not, by itself, prove that customers were dissatisfied or that a retention campaign would increase sales.

### Recommendation 3 — Investigate office-furniture customer experience

**Evidence:** Office furniture had an average review score of 3.64/5 across 1,244 reviewed, delivered orders. Late office-furniture orders averaged 2.35/5, while not-late orders averaged 3.75/5.

**Proposed action:** Review customer feedback for office-furniture orders to identify recurring complaints. Investigate whether product condition, packaging, assembly instructions or seller service are associated with lower ratings.

**How to measure results:** Track the average office-furniture review score and the percentage of reviewed orders with an average rating of 2 or lower. Record the frequency of recurring complaint themes before and after any changes.

**Limitation:** Review scores alone do not identify the causes of dissatisfaction. The proposed investigations should be completed before deciding which operational changes to make.